In [1]:
import os
# os.environ["CUDA_VISIBLE_DEVICES"]="1"
import pandas as pd
import numpy as np
import random
import pickle
import re
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from datetime import datetime
from tqdm import tqdm
from sklearn.metrics import r2_score
import seaborn as sns
# from ucimlrepo import fetch_ucirepo 

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, random_split

from data.data_loader import EPCDataset, PowerWeatherDatasetWithSeason, PowerWeatherDataset, SingleTermDataset, MultiTermDataset
from models.lstm import LSTMModel
from models.lstm_attention import LSTMWithAttention, BiLSTMWithAttention
from models.gru import GRUModel
from models.utils import create_model, train_for_short_term_forecast, train_for_long_term_forecast, load_model, evaluate_for_short_term_forecast, evaluate_for_long_term_forecast

from explainers.utils import get_explainer
# from explainers.lime import LimeExplainer
# from explainers.shap import ShapExplainer
# from explainers.attention import AttentionExplainer
# from explainers.grad_cam import GradCAMExplainer
# from explainers.lrp import LRPExplainer

In [2]:
# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [3]:
# short term 
features_for_1h = [
    'Global_active_power',
    'Global_intensity',
    'Sub_metering_1', 
    'Sub_metering_2', 
    'Sub_metering_3', 
    'Temperature',	
    'Humidity',	
]

# long term
features_for_6h = [
    'Global_active_power',
    'Global_intensity',
    'Sub_metering_1', 
    'Sub_metering_2', 
    'Sub_metering_3', 
    'Temp_Min',	'Temp_Max', 'Temp_Avg',	'Temp_Range',
    'Humidity_Min',	'Humidity_Max',	'Humidity_Avg',	'Humidity_Range'
]

file_path_for_1h = 'data/final_data.csv'
file_path_for_6h = 'data/final_data_per_6hr_with_avg_range.csv'

In [4]:
def load_dataset(params, target_features_long, target_features_short, is_long_term_forecast=True):
    selected_features = dict()
    
    if is_long_term_forecast:
        dataset_6h = EPCDataset(
            # file_path=file_path_for_6h,
            file_path=file_path_for_1h,
            sequence_length=params['long_term_length'],  
            prediction_length=params['long_term_pred_length'],
            target_features=target_features_long
        )
        train_long, train_targets_long, eval_long, eval_targets_long = dataset_6h.load_data()
        
        dataset_1h = EPCDataset(
            file_path=file_path_for_1h,
            sequence_length=params['short_term_length'],  
            prediction_length=params['short_term_pred_length'],
            target_features=target_features_short
        )
        train_short, train_targets_short, eval_short, eval_targets_short = dataset_1h.load_data()
        
        train_data=(train_long, train_short)
        train_targets=(train_targets_long, train_targets_short)
        eval_data=(eval_long, eval_short)
        eval_targets=(eval_targets_long, eval_targets_short)

        selected_features['long'] = dataset_6h.selected_features
        selected_features['short'] = dataset_1h.selected_features

    else:
        dataset_1h = EPCDataset(
            file_path=file_path_for_1h,
            sequence_length=params['sequence_length'],  
            prediction_length=params['prediction_length'],
            target_features=target_features_short
        )
        
        train_sequence, train_targets_sequence, eval_sequence, eval_targets_sequence = dataset_1h.load_data()

        train_data=train_sequence
        train_targets=train_targets_sequence
        eval_data=eval_sequence
        eval_targets=eval_targets_sequence
    
        selected_features['single'] = dataset_1h.selected_features

    return train_data, train_targets, eval_data, eval_targets, selected_features

In [5]:
# Task
## long-term forecast: 일주일 전력량 예측 (7*24) -> Long-term Short-term CNN-LSTM 모델
## Short-term forecast: 하루 전력량 예측 (24)    -> LSTM, GRU, CNN-LSTM 모델 

# 1. 온/습도 정보 없는 케이스도 비교 필요 


dataset_params = {
    # for long-term forecast 

    # 'long_term_length' : 365*4, # 6시간 단위 1년 데이터
    'long_term_length' : 90*24,
    'long_term_pred_length' : 256, # 64, 128, 256
    
    'short_term_length' : 30*24, # 1시간 단위 한달 데이터
    'short_term_pred_length' : 7*24, # 일주일 데이터 예측

    
    # 2. LSTM, GRU, CNN-LSTM  -> 365*24 / 7*24 (비교용)
    # for Short-term forecast 
    'sequence_length' : 24*30,  # -> 일주일 / 10일 / 15일 / 한달  
    'prediction_length' : 24
}

In [6]:
is_long_term_forecast = False

train_data, train_targets, eval_data, eval_targets, selected_features = load_dataset(params=dataset_params,
                                                                                     target_features_long=features_for_1h, 
                                                                                     target_features_short=features_for_1h, 
                                                                                     is_long_term_forecast=is_long_term_forecast)
if isinstance(train_data, tuple):
    print("6-hour data sequences for long-term forecast:")
    print(f"  Train sequences shape: {train_data[0].shape}")
    print(f"  Train targets shape: {train_targets[0].shape}")
    print(f"  Eval sequences shape: {eval_data[0].shape}")
    print(f"  Eval targets shape: {eval_targets[0].shape}")
    print(f"  Selected_features: {selected_features['long']}")
    
    print("\n1-hour data sequences for long-term forecast:")
    print(f"  Train sequences shape: {train_data[1].shape}")
    print(f"  Train targets shape: {train_targets[1].shape}")
    print(f"  Eval sequences shape: {eval_data[1].shape}")
    print(f"  Eval targets shape: {eval_targets[1].shape}")
    print(f"  Selected_features: {selected_features['short']}")
    
else:
    print("1-hour data sequences for short-term forecast:")
    print(f"  Train sequences shape: {train_data.shape}")
    print(f"  Train targets shape: {train_targets.shape}")
    print(f"  Eval sequences shape: {eval_data.shape}")
    print(f"  Eval targets shape: {eval_targets.shape}")
    print(f"  Selected_features: {selected_features['single']}")

1-hour data sequences for short-term forecast:
  Train sequences shape: torch.Size([23925, 720, 13])
  Train targets shape: torch.Size([23925, 24])
  Eval sequences shape: torch.Size([5982, 720, 13])
  Eval targets shape: torch.Size([5982, 24])
  Selected_features: ['Global_active_power', 'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3', 'Temperature', 'Humidity', 'sin_hour', 'cos_hour', 'sin_day', 'cos_day', 'sin_month', 'cos_month']


In [13]:
def load_trained_model(params):
    if params is None:
        return

    # Extract parameters
    model_name = params['model_name']
    hidden_size = params['hidden_size']
    num_layers = params['num_layers']
    dropout = params['dropout']
    num_epochs = params['num_epochs']
    batch_size = params['batch_size']
    learning_rate = params['learning_rate']
    patience = params['patience']
    mse_decay = params['mse_decay']

    # Determine input size and output size
    if is_long_term_forecast == True:
        input_size = {
            'long': len(selected_features['long']),
            'short': len(selected_features['short']),
        }
        output_size = {
            'long': dataset_params['long_term_pred_length'],
            'short': dataset_params['short_term_pred_length'],
        }
    else:
        input_size = {
            'single': len(selected_features['single'])
        }
        output_size = {
            'single': dataset_params['prediction_length']
        }

    if mse_decay:
        mse_alpha = 0.3
        mse_beta = 1.0

    else:
        mse_alpha = 1.0
        mse_beta = 1.0

    # Model 생성 
    model = create_model(
        model_name=model_name,
        input_size=input_size,
        hidden_size=hidden_size,
        num_layers=num_layers,
        output_size=output_size,
        dropout=dropout, 
        long_term_length=dataset_params['long_term_length'], 
        short_term_length=dataset_params['short_term_length']
    ) 

    if 'LS_CNNLSTM' in model_name:
        model_path = './trained_models/{}_long_{}_short_{}_{}_{}_{}_alpha_{}_beta_{}.pth'.format(
            model_name, dataset_params['long_term_length'], dataset_params['short_term_length'],
            dataset_params['long_term_pred_length'], hidden_size, num_layers, mse_alpha, mse_beta
        )
    else:
        model_path = './trained_models/{}_{}_{}_{}_{}.pth'.format(
            model_name, dataset_params['sequence_length'], dataset_params['prediction_length'],
            hidden_size, num_layers
        )
        
    print("Model path:", model_path)

    # Load pre-trained model or train a new one
    if os.path.exists(model_path):
        print(f"Loading the pre-trained {model_name} model...")
        model.load_state_dict(torch.load(model_path))
        model.to(torch.device('cuda' if torch.cuda.is_available() else 'cpu'))
    else:
        print(f"{model_name} model not found. Training a new model...")
        if 'LS_CNNLSTM' in model_name:
            train_for_long_term_forecast(
                model=model,
                model_name=model_name,
                train_data=train_data,
                train_targets=train_targets,
                eval_data=eval_data,
                eval_targets=eval_targets,
                model_path=model_path,
                num_epochs=num_epochs,
                batch_size=batch_size,
                learning_rate=learning_rate,
                patience=patience,
                oversample_eval=True, 
                alpha=mse_alpha, 
                beta=mse_beta
            )
        else:
            train_for_short_term_forecast(
                model=model,
                model_name=model_name,
                train_sequences=train_data,
                train_targets=train_targets,
                eval_sequences=eval_data,
                eval_targets=eval_targets,
                model_path=model_path,
                num_epochs=num_epochs,
                batch_size=batch_size,
                learning_rate=learning_rate,
                patience=patience
            )

    return model, model_path

In [14]:
from itertools import product

In [17]:
model_names = ['LSTM', 'GRU', 'CNNLSTM']
hidden_sizes = [256, 512, 1024]
num_layers = [2, 3]
# mse_decayes = [True]

# results_for_compare = {}
original_results = {} 

# itertools.product를 사용하여 모든 조합 생성
for model_name, hidden_size, num_layer in product(model_names, hidden_sizes, num_layers):
    # print(f"Model: {model_name}, Hidden Size: {hidden_size}, MSE Decay: {mse_decay}")
    model_params = {
        'model_name' : model_name, 
        'hidden_size' : hidden_size,
        'num_layers' : num_layer, 
        'dropout' : 0.3,
        'num_epochs' : 200,
        'batch_size' : 16,
        'learning_rate' : 0.001,
        'patience' : 12,
        'mse_decay' : True
    }

    model, model_path = load_trained_model(model_params)

    print("Evaluating the model...")
    if is_long_term_forecast:
        results = evaluate_for_long_term_forecast(
            model=model,
            eval_data=eval_data,
            eval_targets=eval_targets,
            model_name=model_params['model_name'],
            batch_size=model_params['batch_size'], 
            oversample_eval=True 
        )
    else:
        results = evaluate_for_short_term_forecast(
            model=model,
            eval_sequences=eval_data,
            eval_targets=eval_targets,
            model_name=model_params['model_name'],
            batch_size=model_params['batch_size']
        )

    key = '{}_{}_{}'.format(model_name, hidden_size, num_layer)
    original_results[key] = results

Model path: ./trained_models/LSTM_720_24_256_2.pth
Loading the pre-trained LSTM model...
Evaluating the model...


/tmp/ipykernel_295156/268850128.py:70: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path))
/archive/workspace/XAI/co-work/models/util

R² Score: 0.9893
Adjusted R²: 0.9893
SMAPE: 11.36
MASE: 0.1503
Model path: ./trained_models/LSTM_720_24_256_3.pth
Loading the pre-trained LSTM model...
Evaluating the model...


/tmp/ipykernel_295156/268850128.py:70: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path))
/archive/workspace/XAI/co-work/models/util

R² Score: 0.9912
Adjusted R²: 0.9911
SMAPE: 10.08
MASE: 0.1342
Model path: ./trained_models/LSTM_720_24_512_2.pth
Loading the pre-trained LSTM model...
Evaluating the model...


...

R² Score: 0.9943
Adjusted R²: 0.9943
SMAPE: 7.92
MASE: 0.1029
Model path: ./trained_models/CNNLSTM_720_24_1024_2.pth
Loading the pre-trained CNNLSTM model...
Evaluating the model...


/tmp/ipykernel_295156/268850128.py:70: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path))
/archive/workspace/XAI/co-work/models/util

R² Score: 0.9946
Adjusted R²: 0.9945
SMAPE: 7.69
MASE: 0.0984
Model path: ./trained_models/CNNLSTM_720_24_1024_3.pth
Loading the pre-trained CNNLSTM model...
Evaluating the model...


/tmp/ipykernel_295156/268850128.py:70: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path))
/archive/workspace/XAI/co-work/models/util

R² Score: 0.9964
Adjusted R²: 0.9964
SMAPE: 6.58
MASE: 0.0822


In [19]:
original_results

{'LSTM_256_2': {'R2': 0.9892904758453369,
  'Adjusted R2': 0.9892671474582708,
  'SMAPE': np.float32(11.364356),
  'MASE': np.float32(0.15030493)},
 'LSTM_256_3': {'R2': 0.9911662936210632,
  'Adjusted R2': 0.9911470512981868,
  'SMAPE': np.float32(10.083925),
  'MASE': np.float32(0.13421844)},
 'LSTM_512_2': {'R2': 0.9126116633415222,
  'Adjusted R2': 0.912421306710061,
  'SMAPE': np.float32(25.817215),
  'MASE': np.float32(0.4196061)},
 'LSTM_512_3': {'R2': 0.9957240223884583,
  'Adjusted R2': 0.9957147080940632,
  'SMAPE': np.float32(7.3901176),
  'MASE': np.float32(0.09215691)},
 'LSTM_1024_2': {'R2': 0.9933611750602722,
  'Adjusted R2': 0.993346713812917,
  'SMAPE': np.float32(9.180795),
  'MASE': np.float32(0.114368446)},
 'LSTM_1024_3': {'R2': 0.9962322115898132,
  'Adjusted R2': 0.9962240042759171,
  'SMAPE': np.float32(7.505998),
  'MASE': np.float32(0.08573063)},
 'GRU_256_2': {'R2': 0.979996383190155,
  'Adjusted R2': 0.9799528096280692,
  'SMAPE': np.float32(15.205565),
  '

In [2]:
import re
import json

# 파일 경로
file_path = "important_features_top4.txt"

# 저장할 딕셔너리
important_features_dict = {}

# 정규 표현식 패턴
model_path_pattern = re.compile(r"\./trained_models/(.+)")
feature_pattern = re.compile(r"([\w_]+): ([\d\.]+)")

features = [
    'Global_active_power',
    'Global_intensity',
    'Sub_metering_1', 
    'Sub_metering_2', 
    'Sub_metering_3', 
    'Temperature',    
    'Humidity',    
]


# 변수 초기화
current_model = None
current_section = None

# 파일 읽기
with open(file_path, "r", encoding="utf-8") as file:
    for line in file:
        line = line.strip()
        
        # 모델 경로 감지 및 모델 키 추출
        match = model_path_pattern.match(line)
        if match:
            current_model = match.group(1)
            important_features_dict[current_model] = {"Important Long-term Features": {}, "Important Short-term Features": {}}
            continue

        # 섹션 감지
        if "Important Long-term Features" in line:
            current_section = "Important Long-term Features"
            continue
        elif "Important Short-term Features" in line:
            current_section = "Important Short-term Features"
            continue
        
        # Feature 값 추출
        match = feature_pattern.match(line)
        if match and current_model and current_section:
            feature_name, score = match.groups()
            if feature_name in features:
                important_features_dict[current_model][current_section][feature_name] = float(score)

# important_features_dict


# for key, important_features in important_features_dict.items():

#     # # print(important_features)
#     # model_name = key.split('_long')[0]
#     # hidden_size = 512 if '512' in key else 256
#     # mse_decay = True if '0.3' in key else False

#     important_features_for_long = []
#     important_features_for_short = []
    
#     for feature_long in important_features['Important Long-term Features']:
#         important_features_for_long.append(feature_long)
    
#     for feature_short in important_features['Important Short-term Features']:
#         important_features_for_short.append(feature_short)

In [4]:
important_features_dict

{'LS_CNNLSTM_long_2160_short_720_256_256_3_alpha_0.3_beta_1.0.pth': {'Important Long-term Features': {'Sub_metering_2': 16.752893110308822,
   'Humidity': 14.373554759485302,
   'Global_active_power': 14.342972617265648,
   'Temperature': 14.064040574269374},
  'Important Short-term Features': {'Global_intensity': 15.022332518585879,
   'Global_active_power': 14.99883844462447,
   'Sub_metering_2': 13.884082474862756,
   'Temperature': 13.303119429952611}},
 'LS_CNNLSTM_long_2160_short_720_256_256_3_alpha_1.0_beta_1.0.pth': {'Important Long-term Features': {'Sub_metering_2': 7.521228358542233,
   'Humidity': 7.033322052410343,
   'Temperature': 6.640918862206714,
   'Global_intensity': 6.615907101350656},
  'Important Short-term Features': {'Global_intensity': 16.263148784790687,
   'Temperature': 16.18579343788726,
   'Humidity': 16.071269957950793,
   'Sub_metering_2': 15.798666246736481}},
 'LS_CNNLSTM_long_2160_short_720_256_512_3_alpha_0.3_beta_1.0.pth': {'Important Long-term Feat

In [7]:
# important_features_for_long

In [8]:
def load_dataset_new_features(params, is_long_term_forecast=True):
    selected_features = dict()
    
    if is_long_term_forecast:
        dataset_6h = EPCDataset(
            file_path=file_path_for_1h,
            sequence_length=params['long_term_length'],  
            prediction_length=params['long_term_pred_length'],
            target_features=important_features_for_long
        )
        train_long, train_targets_long, eval_long, eval_targets_long = dataset_6h.load_data()
        
        dataset_1h = EPCDataset(
            file_path=file_path_for_1h,
            sequence_length=params['short_term_length'],  
            prediction_length=params['short_term_pred_length'],
            target_features=important_features_for_short
        )
        train_short, train_targets_short, eval_short, eval_targets_short = dataset_1h.load_data()
        
        train_data=(train_long, train_short)
        train_targets=(train_targets_long, train_targets_short)
        eval_data=(eval_long, eval_short)
        eval_targets=(eval_targets_long, eval_targets_short)

        selected_features['long'] = dataset_6h.selected_features
        selected_features['short'] = dataset_1h.selected_features

    else:
        dataset_1h = EPCDataset(
            file_path=file_path_for_1h,
            sequence_length=params['sequence_length'],  
            prediction_length=params['prediction_length'],
            target_features=important_features_for_1h
        )
        
        train_sequence, train_targets_sequence, eval_sequence, eval_targets_sequence = dataset_1h.load_data()

        train_data=train_sequence
        train_targets=train_targets_sequence
        eval_data=eval_sequence
        eval_targets=eval_targets_sequence
    
        selected_features['single'] = dataset_1h.selected_features

    return train_data, train_targets, eval_data, eval_targets, selected_features

In [9]:
is_long_term_forecast = True

In [9]:
def load_trained_model(params):
    if params is None:
        return

    # Extract parameters
    model_name = params['model_name']
    hidden_size = params['hidden_size']
    num_layers = params['num_layers']
    dropout = params['dropout']
    num_epochs = params['num_epochs']
    batch_size = params['batch_size']
    learning_rate = params['learning_rate']
    patience = params['patience']
    mse_decay = params['mse_decay']

    # Determine input size and output size
    input_size = {
        # 'long': len(important_features_for_6h)+6,
        'long': len(important_features_for_long)+6,
        'short': len(important_features_for_short)+6,
        'single': len(important_features_for_short)+6
    }
    output_size = {
        'long': dataset_params['long_term_pred_length'],
        'short': dataset_params['short_term_pred_length'],
        'single': dataset_params['prediction_length']
    }

    if mse_decay:
        mse_alpha = 0.3
        mse_beta = 1.0

    else:
        mse_alpha = 1.0
        mse_beta = 1.0

    # Model 생성 
    model = create_model(
        model_name=model_name,
        input_size=input_size,
        hidden_size=hidden_size,
        num_layers=num_layers,
        output_size=output_size,
        dropout=dropout, 
        long_term_length=dataset_params['long_term_length'], 
        short_term_length=dataset_params['short_term_length']
    ) 

    if 'LS_CNNLSTM' in model_name:
        model_path = './trained_models/important_features_{}_{}_{}_long_{}_short_{}_{}_{}_{}_alpha_{}_beta_{}.pth'.format(
            input_size['long'], input_size['short'],
            model_name, dataset_params['long_term_length'], dataset_params['short_term_length'],
            dataset_params['long_term_pred_length'], hidden_size, num_layers, mse_alpha, mse_beta
        )
    else:
        model_path = './trained_models/important_features_{}_{}_{}.pth'.format(
            model_name, dataset_params['sequence_length'], dataset_params['prediction_length']
        )
        
    print("Model path:", model_path)

    # Load pre-trained model or train a new one
    if os.path.exists(model_path):
        print(f"Loading the pre-trained {model_name} model...")
        model.load_state_dict(torch.load(model_path))
        model.to(torch.device('cuda' if torch.cuda.is_available() else 'cpu'))
    else:
        print(f"{model_name} model not found. Training a new model...")
        if 'LS_CNNLSTM' in model_name:
            train_for_long_term_forecast(
                model=model,
                model_name=model_name,
                train_data=train_data,
                train_targets=train_targets,
                eval_data=eval_data,
                eval_targets=eval_targets,
                model_path=model_path,
                num_epochs=num_epochs,
                batch_size=batch_size,
                learning_rate=learning_rate,
                patience=patience,
                oversample_eval=True, 
                alpha=mse_alpha, 
                beta=mse_beta
            )
        else:
            train_for_short_term_forecast(
                model=model,
                model_name=model_name,
                train_sequences=train_data,
                train_targets=train_targets,
                eval_sequences=eval_data,
                eval_targets=eval_targets,
                model_path=model_path,
                num_epochs=num_epochs,
                batch_size=batch_size,
                learning_rate=learning_rate,
                patience=patience
            )

    return model

In [ ]:
top_5_resutls = {}


for key, important_features in important_features_dict.items():
    important_features_for_long = []
    important_features_for_short = []

    for feature_long in important_features['Important Long-term Features']:
        important_features_for_long.append(feature_long)
    
    for feature_short in important_features['Important Short-term Features']:
        important_features_for_short.append(feature_short)

    
    train_data, train_targets, eval_data, eval_targets, selected_features = load_dataset_new_features(params=dataset_params,
                                                                                                      is_long_term_forecast=True)

    
    # print(important_features)
    model_name = key.split('_long')[0]
    hidden_size = 512 if '512' in key else 256
    mse_decay = True if '0.3' in key else False

    model_params = {
        'model_name' : model_name, 
        'hidden_size' : hidden_size,
        'num_layers' : 3,  # 3
        'dropout' : 0.3,
        'num_epochs' : 200,
        'batch_size' : 16,
        'learning_rate' : 0.001,
        'patience' : 12,
        'mse_decay' : mse_decay
    }


    # build model 
    model = load_trained_model(model_params)
    
    print("Evaluating the model...")
    if is_long_term_forecast:
        results = evaluate_for_long_term_forecast(
            model=model,
            eval_data=eval_data,
            eval_targets=eval_targets,
            model_name=model_params['model_name'],
            batch_size=model_params['batch_size'], 
            oversample_eval=True 
        )
    else:
        results = evaluate_for_short_term_forecast(
            model=model,
            eval_sequences=eval_short,
            eval_targets=eval_targets_short,
            model_name=model_params['model_name'],
            batch_size=model_params['batch_size']
        )

    key = '{}_{}_{}'.format(model_name, hidden_size, mse_decay)
    top_5_resutls[key] = results

In [13]:
top_5_resutls

{'LS_CNNLSTM_256_True': {'R2 Short': 0.8637720942497253,
  'Adjusted R2': 0.8635428315338884,
  'SMAPE Short': np.float32(31.322771),
  'MASE Short': np.float32(0.53997296)},
 'LS_CNNLSTM_256_False': {'R2 Short': 0.8579791784286499,
  'Adjusted R2': 0.8577401666118014,
  'SMAPE Short': np.float32(31.218555),
  'MASE Short': np.float32(0.54970765)},
 'LS_CNNLSTM_512_True': {'R2 Short': 0.8437401056289673,
  'Adjusted R2': 0.8434771303775855,
  'SMAPE Short': np.float32(32.586624),
  'MASE Short': np.float32(0.5743396)},
 'LS_CNNLSTM_512_False': {'R2 Short': 0.9009826183319092,
  'Adjusted R2': 0.9008159785108589,
  'SMAPE Short': np.float32(88.19094),
  'MASE Short': np.float32(0.48301998)},
 'LS_CNNLSTM_Att_256_True': {'R2 Short': 0.7688463926315308,
  'Adjusted R2': 0.7684573761263667,
  'SMAPE Short': np.float32(37.865715),
  'MASE Short': np.float32(0.70372915)},
 'LS_CNNLSTM_Att_256_False': {'R2 Short': 0.7325351238250732,
  'Adjusted R2': 0.7320849978133349,
  'SMAPE Short': np.fl

In [36]:
original_results.keys() / top_5_resutls

dict_keys(['LS_CNNLSTM_256_True', 'LS_CNNLSTM_256_False', 'LS_CNNLSTM_512_True', 'LS_CNNLSTM_512_False', 'LS_CNNLSTM_Att_256_True', 'LS_CNNLSTM_Att_256_False', 'LS_CNNLSTM_Att_512_True', 'LS_CNNLSTM_Att_512_False'])

In [82]:
first_row = 'Model (w/ Params) \t\t\t\t R2(↑) \t\t\t Adj_R2(↑) \t\t SMAPE(↓) \t\t MASE(↓)'
print(first_row)

for model_type, performance in original_results.items():
    print_str = model_type + '\t\t\t\t'
    for eval_metric, measurement in performance.items():
        print_str += '{: .3f}'.format(measurement) + '\t\t\t'
    print_str += '\n'

    print_str += model_type + '(top 5)' + '\t\t\t'
    top_5_performance = top_5_resutls[model_type]
    for eval_metric, measurement in top_5_performance.items():
        print_str += '{: .3f}'.format(measurement) + '\t\t\t'
    print_str += '\n'

    print(print_str)

Model (w/ Params) 				 R2(↑) 			 Adj_R2(↑) 		 SMAPE(↓) 		 MASE(↓)
LS_CNNLSTM_256_True				 0.860			 0.859			 33.178			 0.553			
LS_CNNLSTM_256_True(top 5)			 0.872			 0.872			 31.659			 0.527			

LS_CNNLSTM_256_False				 0.849			 0.848			 33.692			 0.572			
LS_CNNLSTM_256_False(top 5)			 0.837			 0.837			 33.337			 0.590			

LS_CNNLSTM_512_True				 0.833			 0.833			 33.992			 0.596			
LS_CNNLSTM_512_True(top 5)			 0.903			 0.903			 28.621			 0.462			

LS_CNNLSTM_512_False				 0.835			 0.835			 34.892			 0.596			
LS_CNNLSTM_512_False(top 5)			 0.805			 0.804			 93.898			 0.662			

LS_CNNLSTM_Att_256_True				 0.726			 0.726			 39.989			 0.769			
LS_CNNLSTM_Att_256_True(top 5)			 0.716			 0.716			 40.731			 0.778			

LS_CNNLSTM_Att_256_False				 0.709			 0.708			 41.084			 0.792			
LS_CNNLSTM_Att_256_False(top 5)			 0.781			 0.780			 36.701			 0.683			

LS_CNNLSTM_Att_512_True				 0.778			 0.778			 38.919			 0.697			
LS_CNNLSTM_Att_512_True(top 5)			 0.768			 0.767			 37.424			 0.704			

L

{'LS_CNNLSTM_256_False': {'R2 Short': 0.8371367454528809,
  'Adjusted R2': 0.8368351975989812,
  'SMAPE Short': np.float32(33.336685),
  'MASE Short': np.float32(0.5897087)},
 'LS_CNNLSTM_256_True': {'R2 Short': 0.8719820380210876,
  'Adjusted R2': 0.8717450076252338,
  'SMAPE Short': np.float32(31.65939),
  'MASE Short': np.float32(0.5268928)},
 'LS_CNNLSTM_512_True': {'R2 Short': 0.9033562541007996,
  'Adjusted R2': 0.9031773143255275,
  'SMAPE Short': np.float32(28.621027),
  'MASE Short': np.float32(0.4620556)},
 'LS_CNNLSTM_512_False': {'R2 Short': 0.8045212030410767,
  'Adjusted R2': 0.8041592662010585,
  'SMAPE Short': np.float32(93.898224),
  'MASE Short': np.float32(0.66205746)},
 'LS_CNNLSTM_Att_256_False': {'R2 Short': 0.7808800935745239,
  'Adjusted R2': 0.7804743842712618,
  'SMAPE Short': np.float32(36.70067),
  'MASE Short': np.float32(0.68280476)},
 'LS_CNNLSTM_Att_256_True': {'R2 Short': 0.7161310315132141,
  'Adjusted R2': 0.7156054367222101,
  'SMAPE Short': np.float